# Archivus – Authentication Flow

This notebook covers the **authentication** surface of the Archivus API.

| Step | Endpoint | Method | Auth required |
|------|----------|--------|---------------|
| 1 | `/auth/register` | POST | No |
| 2 | `/auth/login` | POST | No |
| 3 | `/auth/user/info` | GET | Yes |
| 4 | `/auth/drive/invite` | POST | Yes (admin) |
| 5 | `/auth/register` (with invite) | POST | No |
| 6 | `/auth/drive/info` | GET | Yes (admin) |
| 7 | `/auth/drive/remove` | POST | Yes (admin/manager) |
| 8 | `/auth/drive/users` | GET | Yes (admin) |

**Pre-requisites:** start the server with `./archivus server -m home`  
Run cells **in order** — each cell depends on variables set above it.

In [ ]:
import requests, json

BASE = 'http://localhost:8080'

def show(resp):
    """Pretty-print status + JSON body."""
    try:
        body = resp.json()
    except Exception:
        body = resp.text
    print(f'Status : {resp.status_code}')
    print(f'Body   : {json.dumps(body, indent=2)}')
    return body

# State shared across cells
admin_token = None
drive_id    = None
invite_code = None
new_token   = None
new_user_id = None

## 1 · Register an admin user

Business users have two roles:
- **`is_admin: true`** – can create drives and generate invite codes.
- **`is_admin: false`** – must use an invite code to join an existing drive.

A successful registration returns `200 {"message": "user created"}`.  
The server automatically creates a personal drive for admin users.

In [ ]:
resp = requests.post(f'{BASE}/auth/register', json={
    'username':  'samar',
    'password':  'password12',
    'pin':       '123456',
    'email':     'samar@example.com',
    'user_type': 'business',
    'is_admin':  True,
})
show(resp)

## 2 · Login

Login accepts a **6-digit PIN** or a **password** (or both).  
Returns a signed JWT valid for 20 days.  
Pass it as `Authorization: Bearer <token>` on every protected request.

In [ ]:
resp = requests.post(f'{BASE}/auth/login', json={
    'username': 'samar',
    'pin':      '123456',
})
body = show(resp)
admin_token = body.get('token')
print(f'\nadmin_token set: {bool(admin_token)}')

## 3 · Get user info

Returns the authenticated user's profile and every drive they have access to  
(both drives they own and drives they were invited to).

In [ ]:
resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': f'Bearer {admin_token}'})
body = show(resp)

drives  = body.get('drives', [])
drive_id = drives[0]['DriveID'] if drives else None
print(f'\ndrive_id : {drive_id}')
print(f'access   : {drives[0]["AccessLevel"] if drives else None}')

## 4 · Generate an invite code

Only **admin** users may invite others to a drive.

| Access level | Can read | Can write/create folders | Can manage users |
|---|---|---|---|
| `read` | ✓ | ✗ | ✗ |
| `write` | ✓ | ✓ | ✗ |
| `manager` | ✓ | ✓ | ✓ |

Codes expire after **72 hours**.

In [ ]:
resp = requests.post(f'{BASE}/auth/drive/invite',
    headers={'Authorization': f'Bearer {admin_token}'},
    json={
        'drive_id': drive_id,
        'access':   'read',
    })
body = show(resp)
invite_code = body.get('invite_code')
print(f'\ninvite_code : {invite_code}')

## 5 · Register a new user with the invite code

Providing `invite_code` at registration automatically links the new user to  
the invited drive with the access level specified when the code was created.

In [ ]:
resp = requests.post(f'{BASE}/auth/register', json={
    'username':    'newuser2',
    'password':    'newuserpass12',
    'pin':         '654321',
    'email':       'newuser2@example.com',
    'user_type':   'business',
    'is_admin':    False,
    'invite_code': invite_code,
})
show(resp)

## 6 · Login as new user + verify drive access

After logging in, call `/auth/user/info` to confirm the invited drive appears  
with the correct access level.

In [ ]:
resp = requests.post(f'{BASE}/auth/login', json={
    'username': 'newuser2',
    'pin':      '654321',
})
body = show(resp)
new_token = body.get('token')

In [ ]:
resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': f'Bearer {new_token}'})
body = show(resp)

new_user_id = body['user']['ID']
drives      = body.get('drives', [])
print(f'\nnew_user_id : {new_user_id}')
print(f'drives      : {[(d["DriveID"], d["AccessLevel"]) for d in drives]}')

assert any(d['DriveID'] == drive_id for d in drives), \
    'ERROR: invited drive not visible to new user'
print('\n✓ new user can see the invited drive')

## 7 · Get drive info (admin view)

Only admins can call this endpoint.  
Returns the drive's metadata and a per-access-level user breakdown:  
`readUsers`, `writeUsers`, `managerUsers`.

In [ ]:
resp = requests.get(f'{BASE}/auth/drive/info',
    headers={'Authorization': f'Bearer {admin_token}'},
    params={'drive_id': drive_id})
body = show(resp)

read_users = [u['Username'] for u in body.get('readUsers', [])]
print(f'\nread users : {read_users}')
assert 'newuser2' in read_users, 'ERROR: newuser2 not in readUsers'
print('✓ newuser2 is in readUsers')

## 8 · Remove a user from a drive

Who can remove users:
- The drive **owner** (always)
- Users with **manager** access (can remove read/write users but not other managers)

Provide `user_id` (the UUID from `/auth/user/info`) and `drive_id`.

In [ ]:
resp = requests.post(f'{BASE}/auth/drive/remove',
    headers={'Authorization': f'Bearer {admin_token}'},
    json={
        'user_id':  new_user_id,
        'drive_id': drive_id,
    })
show(resp)

In [ ]:
resp = requests.get(f'{BASE}/auth/drive/info',
    headers={'Authorization': f'Bearer {admin_token}'},
    params={'drive_id': drive_id})
body = show(resp)

read_users = [u['Username'] for u in body.get('readUsers', [])]
print(f'\nread users after removal : {read_users}')
assert 'newuser2' not in read_users, 'ERROR: newuser2 still present after removal'
print('✓ newuser2 removed successfully')

## 9 · Get all users in owned drives (admin only)

Returns a list of all drives the caller owns, each with its user breakdown.

In [ ]:
resp = requests.get(f'{BASE}/auth/drive/users',
    headers={'Authorization': f'Bearer {admin_token}'})
show(resp)

## 10 · Edge cases

These cells verify the API rejects invalid requests with the correct status code.

In [ ]:
# No Authorization header → 403
resp = requests.get(f'{BASE}/auth/user/info')
print(f'No auth header : {resp.status_code}  (expect 403)')
assert resp.status_code == 403

In [ ]:
# Malformed / unsigned token → 403
resp = requests.get(f'{BASE}/auth/user/info',
    headers={'Authorization': 'Bearer not-a-real-token'})
print(f'Bad token : {resp.status_code}  (expect 403)')
assert resp.status_code == 403

In [ ]:
# Correct username, wrong PIN → 403
resp = requests.post(f'{BASE}/auth/login', json={
    'username': 'samar',
    'pin':      '000000',
})
print(f'Wrong PIN : {resp.status_code}  (expect 403)')
show(resp)
assert resp.status_code == 403

In [ ]:
# Duplicate username → 400
resp = requests.post(f'{BASE}/auth/register', json={
    'username':  'samar',
    'password':  'password12',
    'pin':       '123456',
    'email':     'other@example.com',
    'user_type': 'business',
    'is_admin':  True,
})
print(f'Duplicate register : {resp.status_code}  (expect 400)')
show(resp)
assert resp.status_code == 400